# 03 — The planning loop (think → act → observe)

**Goal.** Take the single-step exchange from nb 02 and wrap it in a loop so the model can call multiple tools in sequence until it has enough information to answer. Add **hard caps** so the loop cannot run away.

**Why caps matter.** A local 8B model will sometimes get confused and keep calling the same tool over and over. Without a step cap, your terminal hangs and your VRAM stays pinned. With a cap, the loop terminates and you see a clear failure rather than an infinite spin.

**Caps used in this notebook:**
- `MAX_STEPS = 4` — at most 4 tool-calling turns before forcing a final answer
- `MAX_TOOL_OUTPUT_CHARS = 1500` — truncate any single tool result before it goes back to the model
- `MAX_TOTAL_TOOL_CALLS = 6` — across the whole loop, hard ceiling

These are conservative on purpose. With Llama 3.1 8B and a 4K context window, a single oversized tool result will derail the rest of the conversation.

## 1. Setup

Same as nb 02 — re-run nb 01's code cells in this kernel to get the tools and dispatcher.

In [ ]:
import json, nbformat, ollama
from pathlib import Path

nb = nbformat.read(Path("01_tools_not_agents.ipynb"), as_version=4)
for cell in nb.cells:
    if cell.cell_type == "code":
        exec(cell.source, globals())

MODEL = "llama3.1:8b"
MAX_STEPS = 4
MAX_TOOL_OUTPUT_CHARS = 1500
MAX_TOTAL_TOOL_CALLS = 6

print(f"loaded {len(TOOL_SCHEMAS)} tools, model={MODEL}")

## 2. A loop-aware system prompt

We add one line to nb 02's prompt: tell the model it can call multiple tools in sequence and that it should stop when it has enough information.

In [ ]:
SYSTEM_PROMPT = (
    "You are a CTI analyst assistant with access to a small set of tools that "
    "query a CTI corpus (DuckDB) and a public ransomware feed.\n\n"
    "You can call tools in sequence: each tool result will be returned to you "
    "and you can decide what to do next. Stop calling tools as soon as you can "
    "answer the user's question.\n\n"
    "Rules:\n"
    "- Do not call the same tool with the same arguments twice in a row.\n"
    "- If a tool returns an error, try different arguments or stop and explain.\n"
    "- Keep answers short and factual. Cite url_id or alert_id when relevant."
)

## 3. The loop

On each iteration:

1. Send the conversation to Ollama with `tools=`.
2. If the model emits `tool_calls`, dispatch each one, append a `tool` message per call.
3. If the model returns plain content (no tool calls), we're done.
4. If we've hit `MAX_STEPS` or `MAX_TOTAL_TOOL_CALLS`, force a final answer with one more call that doesn't allow tools.

Notice: we truncate tool results before they go back into the messages list. This is the single most important detail for keeping a small model on track.

In [ ]:
def truncate_for_model(payload, max_chars=MAX_TOOL_OUTPUT_CHARS):
    """Serialize a tool result and clip it so it fits comfortably in context.
    Append a sentinel so the model knows it was cut."""
    s = json.dumps(payload)
    if len(s) <= max_chars:
        return s
    return s[:max_chars] + f"... [truncated, total {len(s)} chars]"

def run_agent(question, *, model=MODEL, verbose=True):
    """Run a planning loop until the model stops requesting tools or caps trigger.
    Returns (final_text, trace) where trace is a list of step dicts useful for nb 06."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    trace = []
    total_tool_calls = 0

    for step in range(1, MAX_STEPS + 1):
        if verbose:
            print(f"\n=== step {step} ===")

        resp = ollama.chat(
            model=model,
            messages=messages,
            tools=TOOL_SCHEMAS,
            options={"temperature": 0.2, "num_ctx": 4096},
        )
        msg = resp["message"]
        messages.append(msg)

        tool_calls = msg.get("tool_calls") or []
        step_record = {"step": step, "tool_calls": [], "content": msg.get("content") or ""}

        if not tool_calls:
            if verbose:
                print("  (no tool call — model is answering)")
                print("  content:", step_record["content"][:400])
            trace.append(step_record)
            return step_record["content"], trace

        for tc in tool_calls:
            fn_name = tc["function"]["name"]
            fn_args = tc["function"].get("arguments") or {}
            if isinstance(fn_args, str):
                try:
                    fn_args = json.loads(fn_args)
                except json.JSONDecodeError:
                    fn_args = {}

            total_tool_calls += 1
            if total_tool_calls > MAX_TOTAL_TOOL_CALLS:
                if verbose:
                    print(f"  [cap hit: total tool calls > {MAX_TOTAL_TOOL_CALLS}]")
                step_record["tool_calls"].append({"name": fn_name, "args": fn_args, "skipped": True})
                trace.append(step_record)
                return _force_final_answer(messages, model, verbose), trace

            result = dispatch_tool_call(fn_name, fn_args)
            tool_content = truncate_for_model(result)
            messages.append({"role": "tool", "name": fn_name, "content": tool_content})

            step_record["tool_calls"].append({"name": fn_name, "args": fn_args,
                                              "result_preview": tool_content[:200]})
            if verbose:
                print(f"  -> {fn_name}({fn_args})")
                print(f"     {tool_content[:200]}")

        trace.append(step_record)

    # Hit MAX_STEPS without the model giving up tools — force a final answer.
    if verbose:
        print(f"\n[cap hit: reached MAX_STEPS={MAX_STEPS}]")
    return _force_final_answer(messages, model, verbose), trace

def _force_final_answer(messages, model, verbose):
    """Make one more call without tools, instructing the model to wrap up."""
    messages = messages + [{
        "role": "user",
        "content": "Stop calling tools. Based on what you have, give your best final answer in 1-3 sentences.",
    }]
    resp = ollama.chat(model=model, messages=messages,
                       options={"temperature": 0.2, "num_ctx": 4096})
    final = resp["message"].get("content") or "(no content)"
    if verbose:
        print("--- forced final ---\n ", final[:400])
    return final

## 4. Demo questions

Two simple ones first, then one that genuinely needs a chain (find a doc → fetch its text).

In [ ]:
answer, trace = run_agent("Are there any high-severity alerts? If yes, list their titles.")
print("\n=== FINAL ===\n", answer)
print("\n=== STEPS ===", len(trace))

In [ ]:
answer, trace = run_agent(
    "Find one document in the corpus labeled 'Hacking', then summarize its content."
)
print("\n=== FINAL ===\n", answer)
print("\n=== STEPS ===", len(trace))

In [ ]:
# Stress test: a question with no good answer in the corpus. Watch how the model handles it.
answer, trace = run_agent(
    "Has the BlackCat ransomware group posted any new victims in the last 7 days? "
    "Give a yes or no with the count."
)
print("\n=== FINAL ===\n", answer)
print("\n=== STEPS ===", len(trace))

## 5. What you'll see, and why

The chain in question 2 is the most informative. A working trace looks roughly like:

1. `query_documents(label='Hacking')` → returns 1 url_id
2. `get_document(url_id=...)` → returns truncated text
3. (no tool call) → final summary

Common failures with an 8B model:

- **Skipping the chain** — model summarises from `query_documents`'s metadata alone, never calls `get_document`. Final answer is shallow.
- **Re-querying** — model calls `query_documents` twice with the same args. The system prompt explicitly forbids this; small models still do it.
- **Cap hit** — model can't decide to stop; we force a final answer.

All three are real, all three are normal. nb 06 logs traces and lets you replay them.

## What's next

nb 04 introduces **memory** — the loop above starts fresh every call. The agent forgets. We'll show how to use DuckDB as long-term memory and a rolling summary as short-term memory, so the agent can answer follow-up questions without re-doing work.